# test — 自动化数据采集（pipeline 测试）

配套 `tool/uload.ipynb` 生成的 `cfg_<kind>_u.._m.._s<seed>.json`：
对每份 cfg，以 workers=m 起 `bin/client` + `bin/server` 灌 cfg，跑 `dur_s` 秒 → 终止 →
把 client 导出的 trace（`tool/temp/tracing.csv`）**复制**到结果目录（原件保留在 tool/temp，不删）。**只搬运原始 trace，不算指标**；
**默认每份测一轮(trials=1)**。

## 结果存放（镜像输入目录）
- 输入目录 = `cfg_dir`（默认 `tool/uload/`，可递归含子目录）；
- 结果根 = `test_dir`（默认 `tool/test/`），镜像 `cfg_dir` 的子目录结构；
- 每份 cfg → 同目录 `trace_<cfg名去cfg_前缀>.csv`；`trials>1` 加 `_r<rep>`。

## tool/temp（agent 导出中间目录）
- client(agent) 把 trace 写 `./tool/temp/tracing.csv`（宏 `FINS_EXPORT_TRACING_PATH`，相对启动 cwd=仓库根；
  `dag.json` 同目录）；client 启动时会自建该目录；
- notebook 用 `temp_dir` 指向它（默认 `tool/temp`，相对仓库根或绝对）；跑前记下基线，
  跑后只把**本轮新写入**的 trace **复制**到结果目录归档——tool/temp 原件绝不删除。

In [3]:
# ==== 库体：test() —— 跑 cfg → 把 tool/temp/tracing.csv 复制到 out（镜像子目录，与 cfg 同名）====
import os, re, time, shutil, signal, subprocess

CFG_RE = re.compile(r"cfg_(?P<kind>\w+)_u(?P<u>\d+)_m(?P<m>\d+)_s(?P<seed>\d+)\.json$")
GRACE = 20    # SIGTERM 后等 client 正常退出的秒数；超时才 SIGKILL(视为非正常退出)

def repo_root():
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, "bin", "client")) and os.path.isdir(os.path.join(d, "tool")):
            return d
        p = os.path.dirname(d)
        if p == d: return None
        d = p

def find_cfgs(directory):
    out = []
    for rt, _, fns in os.walk(directory):
        for fn in sorted(fns):
            mt = CFG_RE.match(fn)
            if mt:
                out.append((os.path.relpath(os.path.join(rt, fn), directory),
                            int(mt["u"]) / 100.0, int(mt["m"]),
                            mt["kind"], int(mt["seed"])))
    return out

def run_and_cp(cfg_path, workers, dur_s, warm_s, port, dest, root, cores=None, temp_dir="tool/temp"):
    """起 client+server 灌 cfg → 跑 dur_s → SIGTERM → 把 tool/temp 里 client 导出的 trace 复制到 dest。

    client 把 trace 写 ./tool/temp/tracing.csv（宏 FINS_EXPORT_TRACING_PATH，相对启动 cwd=仓库根；
    dag.json 同理写 ./tool/temp/），temp_dir 即该导出目录（仓库根下相对名/绝对路径）。client 自建目录，
    这里仍先建(幂等)。搬运用 copy2 且不删 tool/temp 原件——以跑前基线 vs 跑后
    size/mtime 判定“本轮新写入”，旧残留/失败轮不误收也不清除。

    ★ 退出确认（回答“client 是否正常退出”）：收尾统一 先 SIGTERM → 限时 GRACE 秒等进程自然退出，
    拿退出码 rc。SIGTERM 会走 client 的信号处理（置停止位→teardown→rc=0），teardown 里写 trace：
      · rc==0 且未超时被强杀 ⇒ client 正常退出（trace 已导出），本轮才算有效、才去收文件；
      · 超时被 SIGKILL / rc≠0 ⇒ 非正常退出，无干净 trace，返回 False(rc, killed)；
      · 全程 try/finally 语义：即便 Ctrl-C(KeyboardInterrupt) 也会先停掉进程组、再收掉 cpuset
        分区，然后才把中断抛回（不会留孤儿 client / 占核分区）。

    cores=None → 裸 client（开发自检）；cores 如 "1-6" → sudo tool/agent.sh <cores> <workers>（需 root/密码）。
    返回 (ok, rc, killed)：ok=采到并复制归档；rc=client 退出码(正常 0)；killed=是否靠 SIGKILL 收场。"""
    export_dir = temp_dir if os.path.isabs(temp_dir) else os.path.join(root, temp_dir)
    os.makedirs(export_dir, exist_ok=True)
    cands = [os.path.join(export_dir, "tracing.csv"),
             os.path.join(root, "tracing.csv")]         # 旧二进制：导出根目录（兜底）
    # 跑前记基线（size+mtime_ns）——绝不删除 tool/temp 里的原件；跑后只收“本轮新写入”。
    def _sig(p):
        try:
            st = os.stat(p); return (st.st_size, st.st_mtime_ns)
        except FileNotFoundError:
            return None
    snap = {c: _sig(c) for c in cands}
    if cores:
        cmd = ["sudo", os.path.join(root, "tool", "agent.sh"), str(cores), str(workers)]
    else:
        cmd = [os.path.join(root, "bin", "client"), str(port), os.path.join(root, "lib"),
               str(workers)]   # 插件 .so 现生成在 lib/
    cl = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                          cwd=root, start_new_session=True)
    intr = None
    try:
        time.sleep(warm_s)
        srv = subprocess.run([os.path.join(root, "bin", "server"), cfg_path, str(port)],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=root)
        if srv.returncode != 0:               # cfg 没送进去 → 提示并跳过 dur_s 白等，直接走收尾
            print(f"      [run] server 灌 cfg 失败 rc={srv.returncode}: {(srv.stdout or '').strip()}")
        else:
            time.sleep(dur_s)
    except BaseException as ex:               # 含 KeyboardInterrupt：收尾完再抛
        intr = ex
    # ── 统一收尾：SIGTERM 优雅退出 → 限时等自然退出；超时才 SIGKILL ──
    killed = False
    try:
        os.killpg(cl.pid, signal.SIGTERM)     # client: SIGTERM → 置停止位 → teardown(写 trace) → rc=0
    except ProcessLookupError:
        pass                                  # 进程组已不在(早崩/已退出)：下面 wait 拿真实 rc
    try:
        rc = cl.wait(timeout=GRACE)
    except subprocess.TimeoutExpired:
        killed = True
        try: os.killpg(cl.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        rc = cl.wait()
    if cores:                                 # 无论成败都收掉独占分区（幂等）
        subprocess.run(["sudo", os.path.join(root, "tool", "agent.sh"), "-r"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, cwd=root)
    if intr is not None:
        raise intr                            # 清理完再抛（Ctrl-C 等）
    if killed or rc != 0:
        return False, rc, killed              # 非正常退出：没走 teardown，无干净 trace
    for _ in range(30):                       # 正常退出：trace 已写完，收“本轮新写入”那份
        for src in cands:
            cur = _sig(src)
            if cur is not None and cur != snap.get(src):
                try:
                    with open(src) as f:      # 只有表头 → 没跑出任务(cfg 没灌入等)，不归档
                        n_data = sum(1 for ln in f if ln.strip() and not ln.startswith("tid,"))
                except OSError:
                    time.sleep(0.1); continue
                if n_data < 1:
                    return False, rc, killed  # rc=0 正常退出但无 trace(原件留在 tool/temp，不删)
                try:
                    shutil.copy2(src, dest)   # 复制归档，tool/temp 原件保留
                    return True, rc, killed
                except (FileNotFoundError, PermissionError, OSError):
                    time.sleep(0.1)
        time.sleep(0.1)
    return False, rc, killed
def test(directory, out, temp_dir="tool/temp", dur_s=5.0, warm_s=1.0, trials=1,
         only=None, u=None, m=None, port=18080, cores=None):
    """主入口：对 directory 里每份 cfg 跑一轮(dur_s 可配)，把 temp_dir 里的 tracing.csv 复制到(原件保留)
    out/<镜像子目录>/trace_<cfg名去cfg_前缀>.csv（trials>1 加 _r<rep>）。cores=None 裸 client；
    cores="1-6" 走 sudo tool/agent.sh 独占核。返回成功搬了几份。"""
    root = repo_root()
    if root is None: raise RuntimeError("找不到仓库根(含 client 与 tool/)")
    directory = directory if os.path.isabs(directory) else os.path.join(root, directory)
    out = out if os.path.isabs(out) else os.path.join(root, out)
    cfgs = find_cfgs(directory)
    if only: cfgs = [c for c in cfgs if c[3] in only]
    if u is not None: cfgs = [c for c in cfgs if abs(c[1] - u) < 1e-9]
    if m is not None: cfgs = [c for c in cfgs if c[2] == m]
    if not cfgs:
        print(f"[test] {directory} 下没找到 cfg_*.json（先到 tool/uload.ipynb 生成）")
        return 0
    moved = 0
    for rel, uu, mm, kind, seed in cfgs:
        base = os.path.splitext(os.path.basename(rel))[0]
        rdir = os.path.dirname(rel)
        d = out if rdir == "." else os.path.join(out, rdir)
        os.makedirs(d, exist_ok=True)
        stem = base[4:] if base.startswith("cfg_") else base   # 去掉 cfg_ 前缀
        for rep in range(1, trials + 1):
            suffix = "" if trials == 1 else f"_r{rep}"
            dest = os.path.join(d, "trace_" + stem + suffix + ".csv")   # trace_xxx.csv
            ok, rc, killed = run_and_cp(os.path.join(directory, rel), mm, dur_s, warm_s, port, dest, root,
                                        cores=cores, temp_dir=temp_dir)
            if ok:       why = f"正常退出 rc={rc}"
            elif killed: why = f"超时被SIGKILL rc={rc}"
            elif rc == 0: why = "正常退出但无 trace(见上方 [run] 提示 / cfg 未灌入?)"
            elif rc is not None: why = f"退出码≠0 rc={rc}"
            else:        why = "rc=?"
            print(f"[test] {kind:9s} u={uu} m={mm} seed={seed} r{rep} -> "
                  + ("moved " + os.path.relpath(dest) + " (" + why + ")" if ok
                     else "no_trace (" + why + ")"))
            moved += int(ok)
    print(f"[test] 完成: 搬了 {moved}/{len(cfgs) * trials} 份 trace → {out}")
    return moved

### 用法
`test(directory, out='tool/test', temp_dir='tool/temp', dur_s=5.0, warm_s=1.0, trials=1, only, u, m, port, cores=None)`：
- 对每份 cfg：起 client → 预热 `warm_s` → `server` 灌 cfg → 跑 **`dur_s` 秒** → SIGTERM →
  把 client 在 `temp_dir` 导出的 `tracing.csv` **复制**到 `out` 镜像子目录为 `trace_<cfg名去cfg_前缀>.csv`（原件保留）；
- **独占核(正式实验)：`cores="1-6"` → 走 `sudo tool/agent.sh <cores> <workers>`**（需要 root；jupyter 里要
  passwordless sudo，否则请在终端 `sudo` 跑驱动脚本）；`cores=None` → 裸 `bin/client`，仅开发自检；
- `directory/out/temp_dir` 相对路径按仓库根解析；client 启动 cwd=仓库根 → 导出目录固定 `root/<temp_dir>`。

In [4]:
# ── 参数（可配置）────────────────────────────
cfg_dir   = "tool/uload/"   # 目标 json(cfg)存放目录
test_dir = "tool/test/"          # 结果根(镜像 cfg_dir 子目录结构)
temp_dir = "tool/temp/"              # agent(client)导出目录(./tool/temp/tracing.csv，相对仓库根)
dur_s     = 2.0                  # ★ 测试时长(秒)：每份 cfg 跑多久
warm_s    = 1                  # 启动 client 后/计时前预热
trials    = 1                    # 每份测几轮(默认 1)

# ── 跑并复制：起 client/server→跑 dur_s→终止→把 tool/temp/tracing.csv 复制到 test 镜像(原件保留)，
#    每个 cfg 生成与它同名的 trace_*.csv（纯搬运，不算指标）────────
test(cfg_dir, test_dir, temp_dir, dur_s=dur_s, warm_s=warm_s, trials=trials)

[test] fork      u=0.7 m=1 seed=10000 r1 -> moved test/trace_fork_u70_m1_s10000.csv (正常退出 rc=0)
[test] fork      u=0.7 m=2 seed=10001 r1 -> moved test/trace_fork_u70_m2_s10001.csv (正常退出 rc=0)
[test] fork      u=0.7 m=3 seed=10002 r1 -> moved test/trace_fork_u70_m3_s10002.csv (正常退出 rc=0)
[test] fork      u=0.8 m=1 seed=10003 r1 -> moved test/trace_fork_u80_m1_s10003.csv (正常退出 rc=0)
[test] fork      u=0.8 m=2 seed=10004 r1 -> moved test/trace_fork_u80_m2_s10004.csv (正常退出 rc=0)
[test] fork      u=0.8 m=3 seed=10005 r1 -> moved test/trace_fork_u80_m3_s10005.csv (正常退出 rc=0)
[test] fork      u=0.9 m=1 seed=10006 r1 -> moved test/trace_fork_u90_m1_s10006.csv (正常退出 rc=0)
[test] fork      u=0.9 m=2 seed=10007 r1 -> moved test/trace_fork_u90_m2_s10007.csv (正常退出 rc=0)
[test] fork      u=0.9 m=3 seed=10008 r1 -> moved test/trace_fork_u90_m3_s10008.csv (正常退出 rc=0)
[test] join      u=0.7 m=1 seed=10000 r1 -> moved test/trace_join_u70_m1_s10000.csv (正常退出 rc=0)
[test] join      u=0.7 m=2 seed=10001 r1

45